# Guía y Pistas: Ejercicio Integrador — Clase 2

Esta guía acompaña el ejercicio integrador de la Clase 2. Está organizada en el mismo orden en que aparecen los pasos en el notebook principal.

> **Cómo usar esta guía:** Intentá resolver cada paso por tu cuenta antes de abrir la pista correspondiente. El objetivo no es copiar el código, sino entender qué hace cada parte y por qué.

---

## Parte 1: La función `analizar_texto()`

Esta función es el núcleo de la aplicación. Recibe el texto ingresado por el usuario, lo procesa con la clase `ProcesadorTexto` y devuelve dos valores: un string con el resumen y un diccionario con las frecuencias.

### Resolución
A continuación, cada bloque queda completo para que el notebook pueda ejecutarse de principio a fin sin celdas incompletas.

### Paso 1 — Crear una instancia de `ProcesadorTexto`

Para usar los métodos de la clase, primero necesitás crear un **objeto** a partir de ella. Eso se llama **instanciar** la clase.

La sintaxis es: `nombre_del_objeto = NombreDeClase(argumentos)`

In [ ]:
import re
from collections import Counter

class ProcesadorTexto:
    """Clase para limpiar y analizar texto de forma simple."""

    def __init__(self, texto):
        self.texto_original = texto
        self.texto_limpio = re.sub(r"[^\w\sáéíóúüñ]", " ", texto.lower())
        self.palabras = [p for p in self.texto_limpio.split() if p]

    def contar_palabras(self):
        return len(self.palabras)

    def contar_palabras_unicas(self):
        return len(set(self.palabras))

    def calcular_frecuencia(self):
        return dict(Counter(self.palabras))


### Paso 2 — Obtener métricas usando los métodos del objeto

Una vez que tenés el objeto, podés llamar a sus métodos con la notación `objeto.metodo()`. Cada método devuelve un valor que podés guardar en una variable.

In [ ]:
def analizar_texto(texto):
    """Procesa un texto y devuelve resumen + frecuencias para Gradio."""
    procesador = ProcesadorTexto(texto)

    total_palabras = procesador.contar_palabras()
    palabras_unicas = procesador.contar_palabras_unicas()
    frecuencias = procesador.calcular_frecuencia()

    if frecuencias:
        palabra_mas_frecuente = max(frecuencias, key=frecuencias.get)
        repeticiones = frecuencias[palabra_mas_frecuente]
    else:
        palabra_mas_frecuente = "N/A"
        repeticiones = 0

    resumen = (
        f"--- Resumen del Análisis ---\n"
        f"Total de palabras:     {total_palabras}\n"
        f"Palabras únicas:       {palabras_unicas}\n"
        f"Palabra más frecuente: '{palabra_mas_frecuente}' ({repeticiones} veces)"
    )
    return resumen, frecuencias


### Paso 3 — Encontrar la palabra más frecuente (opcional)

La función `max()` puede recibir un argumento `key` que le indica **con qué criterio** comparar los elementos. En un diccionario, si pasás el diccionario como iterable, `max()` itera sobre sus claves.

Al usar `key=frecuencias.get`, cada clave se compara por su valor asociado en el diccionario.

Es importante verificar antes que el diccionario no esté vacío (podría serlo si el usuario ingresa solo espacios en blanco).

In [ ]:
# Bloque de lógica de frecuencia (referencia del paso 3)
frecuencias_demo = {"python": 3, "pln": 1}
palabra_mas_frecuente = max(frecuencias_demo, key=frecuencias_demo.get)
repeticiones = frecuencias_demo[palabra_mas_frecuente]
print(palabra_mas_frecuente, repeticiones)


### Paso 4 — Formatear el resumen

El resumen es el texto que va a ver el usuario en la interfaz. Usá un **f-string** para insertar las variables dentro del texto.

Los paréntesis alrededor del string permiten dividirlo en varias líneas sin necesidad de poner el f-string todo junto: Python une automáticamente los strings literales adyacentes.

In [ ]:
# Ejemplo aislado del formato del resumen (paso 4)
resumen_demo = (
    f"--- Resumen del Análisis ---\n"
    f"Total de palabras:     {4}\n"
    f"Palabras únicas:       {3}\n"
    f"Palabra más frecuente: '{'python'}' ({2} veces)"
)
print(resumen_demo)


### Paso 5 — Devolver los resultados

En Python, una función puede devolver múltiples valores separados por comas. Gradio los asigna automáticamente a cada componente de salida en el orden en que están declarados en `outputs`.

In [ ]:
# Prueba final de la función completa (paso 5)
print(analizar_texto("Python para PLN, Python para análisis." )[0])


---

## Parte 2: La interfaz de Gradio

Gradio conecta tu función Python con una interfaz web. Solo necesitás indicarle cuál es la función, qué recibe y qué muestra.

### Paso 6 — Asignar la función al parámetro `fn`

El parámetro `fn` recibe la función que Gradio va a ejecutar cada vez que el usuario presione el botón de envío.

> Importante: se pasa el **nombre** de la función, sin paréntesis. Los paréntesis ejecutan la función; sin paréntesis, solo se pasa la referencia.

In [ ]:
import gradio as gr

# Gradio ejecuta esta función cada vez que se presiona "Analizar".
fn_analisis = analizar_texto


### Paso 7 — Configurar las salidas

El parámetro `outputs` recibe una lista de componentes de Gradio, uno por cada valor que devuelve tu función.

Como la función devuelve **dos valores** (un string y un diccionario), necesitás **dos componentes**:
- `gr.Textbox` para mostrar texto plano.
- `gr.JSON` para mostrar un diccionario con formato visual interactivo.

In [ ]:
demo = gr.Interface(
    fn=fn_analisis,
    inputs=gr.Textbox(lines=8, label="Texto a analizar"),
    outputs=[
        gr.Textbox(label="Resumen del Análisis"),
        gr.JSON(label="Frecuencia de Palabras")
    ],
    title="Analizador básico de texto",
    description="Aplicación simple para medir volumen, vocabulario y frecuencia de términos."
)


### Paso 8 — Lanzar la interfaz

El método `.launch()` inicia el servidor local y abre la interfaz.

El argumento `share=True` crea un enlace público temporal (72 hs) para compartir la aplicación. Requiere conexión a internet. En entorno local, `inbrowser=True` abre el navegador automáticamente.

In [ ]:
# Elegí la opción según tu entorno:
# demo.launch()                # Entorno local / notebooks sin enlace público
# demo.launch(share=True)      # Enlace público temporal


---

## Consejos para depurar el código

**Probá por partes antes de conectar con Gradio.**
Antes de lanzar la interfaz, ejecutá tu función directamente en una celda para verificar que devuelve lo esperado:

In [ ]:
# Prueba directa de la función sin interfaz
texto_prueba = "Python es un lenguaje poderoso. Python se usa en PLN."
resumen_prueba, frecuencias_prueba = analizar_texto(texto_prueba)
print(resumen_prueba)
print(frecuencias_prueba)


**Usá `print()` para inspeccionar variables intermedias.**
Si algo no funciona, agregá prints temporales dentro de la función para ver qué valor tiene cada variable en cada paso:

In [ ]:
# Verificación rápida de tipos y consistencia
print(f"Total palabras: {sum(frecuencias_prueba.values())}")
print(f"Tipo de frecuencias: {type(frecuencias_prueba)}")


**Leé los mensajes de error con atención.**
Gradio muestra los errores en rojo en la consola o en la interfaz. El mensaje indica el tipo de error y la línea donde ocurrió. Podés copiar ese mensaje completo y preguntarle a la IA:
> *"Tengo este error al ejecutar mi función con Gradio: [pegá el error]. ¿Qué puede estar causándolo?"*

**Si algo falla de forma inexplicable**, reiniciá el runtime de Colab: `Runtime → Restart runtime` y volvé a ejecutar las celdas en orden.

---

## Extensión opcional para estudiantes que terminaron antes

Si terminaste el ejercicio y querés un desafío adicional, podés ampliar la clase `ProcesadorTexto` con los siguientes métodos.

Intentá implementarlos vos mismo antes de mirar el código. Usá la IA para explorar alternativas o pedir explicaciones de las funciones que no conozcas.

**Opción A — Longitud media de palabras**

In [ ]:
def calcular_longitud_media(texto):
    """Calcula la longitud promedio de las palabras de un texto."""
    proc = ProcesadorTexto(texto)
    if not proc.palabras:
        return 0
    total_letras = sum(len(palabra) for palabra in proc.palabras)
    return total_letras / len(proc.palabras)


**Opción B — Las 5 palabras más frecuentes**

`Counter` es una subclase de diccionario de la librería estándar de Python, especializada en conteos. El método `.most_common(n)` devuelve los `n` elementos más frecuentes en orden descendente.

In [ ]:
def top_palabras(texto, n=5):
    """Devuelve las n palabras más frecuentes como lista de tuplas."""
    proc = ProcesadorTexto(texto)
    return Counter(proc.palabras).most_common(n)


**Opción C — Diversidad léxica**

La diversidad léxica es el porcentaje de palabras únicas sobre el total. Un índice alto indica vocabulario variado; uno bajo sugiere repetición. Es una métrica usada en análisis de estilo y legibilidad.

In [ ]:
def calcular_diversidad_lexica(texto):
    """Devuelve el porcentaje de palabras únicas sobre el total (0 a 100)."""
    proc = ProcesadorTexto(texto)
    total = proc.contar_palabras()
    if total == 0:
        return 0
    return (proc.contar_palabras_unicas() / total) * 100


---

## Texto de prueba

Podés usar el texto de abajo para verificar que tu aplicación funciona correctamente antes de probarla con textos propios.

In [ ]:
texto_de_prueba = """
Python es un lenguaje de programación versátil y poderoso.
Python se utiliza en muchas áreas, desde desarrollo web hasta ciencia de datos.
La simplicidad de Python lo hace ideal para principiantes,
pero su potencia satisface incluso a programadores expertos.
"""

resumen, frecuencias = analizar_texto(texto_de_prueba)
print(resumen)
print("Top 5 palabras:", top_palabras(texto_de_prueba, n=5))
print("Longitud media:", round(calcular_longitud_media(texto_de_prueba), 2))
print("Diversidad léxica:", round(calcular_diversidad_lexica(texto_de_prueba), 2), "%")


---

> Los errores y la experimentación son parte fundamental del proceso de aprendizaje. Si la aplicación no funciona en el primer intento, releé el mensaje de error, identificá en qué paso falla y ajustá desde ahí.